# Question tables for the paper (Appendix B)

Generates the LaTeX for the two question tables, straight from the released
qa_jsons, so the question wording, the choices offered, which figure types
each question is asked of, and the example answers cannot drift from the data:

* **Table 1** (`tab:questions`) -- example questions per level for our figures
  (contour, GMM-sky, real-sky), plus the published figures where they share a
  question.
* **Table 2** (`tab:published_questions`) -- every question asked of the
  published (Astronomy Image Explorer) figures, where its ground truth comes
  from, and its range over the 104 human-verified figures.

The last section recomputes the numbers quoted in the angular-extent appendix
(Appendix C): how closely the two angular-width definitions -- ours and the
published figures' -- agree.

Each table is printed and written to `paper_figures/` as a `.tex` fragment.
The paper keeps its own copy of the table (so it builds without this repo);
after a change here, paste the printed table over the one in `paper_draft.tex`.

In [1]:
# ======================== CONFIG ========================
data_root    = '~/astro_sky_image_vqa/'
qa_jsons_dir = data_root + 'VQA_full/qa_jsons/'
realsky_dir  = data_root + 'VQA_realskyimages/'
published_run = data_root + 'LMM_outputs_realskyimages/gemini/'
tab_out_dir  = '~/Dropbox/WASP2026/paper_figures/'

# Table 1 rows, in order: (level, question key, example figure, abridge,
# answer format).  `abridge` is None (keep the question whole), a phrase (cut
# after it and append [...]), or a (phrase, resume) pair (cut after `phrase`,
# then pick the text up again at `resume`).  The answer format turns the stored
# ground truth into the string printed in the table.
table1_rows = [
    (1, 'plot types (list)',                  'vqa_000001', None, str),
    (1, 'colormap',                           'vqa_000001', None, str),
    (1, 'epoch',                              'vqa_000004', None, str),
    (1, 'finest unit right ascension',        'vqa_000001', None, str),
    (1, 'minimum right ascension axis limit', 'vqa_000001', 'hours, minutes and seconds', str),
    (1, 'image or lines (list)',              'vqa_000007', None, str),
    (2, 'field height',                       'vqa_000002', None, '{:.2f}'.format),
    (2, 'field width',                        'vqa_000002', None, '{:.2f}'.format),
    (2, 'mean color',                         'vqa_000001', None, '{:.1f}'.format),
    (3, 'distribution-image + list)',         'vqa_000002', None, str),
    (3, 'distribution-color + list)',         'vqa_000007', None, str),
    (3, 'pixel scale',                        'vqa_000001', None, '{:.1f}'.format),
    (3, 'center right ascension',             'rsky_000001', ('J2000 (FK5) frame?', 'Give the value'),
     '{:.2f}'.format),
]

In [2]:
import os, re, json, glob, pickle, collections
import numpy as np

ex = os.path.expanduser
os.makedirs(ex(tab_out_dir), exist_ok=True)

CATEGORY_NAMES = {'contour': 'contour', 'sky-gmm': 'GMM-sky', 'sky-real': 'real-sky',
                  'published': 'published'}


def entries(rec):
    """(name, entry) for every question in a qa_json record."""
    for lvl in rec['VQA'].values():
        for items in lvl.values():
            for name, e in items.items():
                yield name, (e if 'Q' in e else e['plot0'])


def answer(entry, name):
    """The ground-truth value, unwrapped from its dict / one-item list."""
    a = entry['A']
    while isinstance(a, dict):
        a = a[name] if name in a else next(iter(a.values()))
    return a[0] if isinstance(a, list) and len(a) == 1 else a


CHOICES = re.compile(r'\s*Please choose (?:the [\w ]+?|your answer) from the following list: \[([^\]]*)\]\.?')

def question_text(entry, abridge=None):
    """Question as printed: choices pulled out into [..], long ones abridged."""
    q = entry['question'].strip()
    choices = CHOICES.search(entry.get('context', '') + ' ' + q)
    q = CHOICES.sub('', q).strip()
    if abridge:
        cut, resume = abridge if isinstance(abridge, tuple) else (abridge, None)
        head = q[:q.index(cut) + len(cut)] + ' [...]'
        q = head + (' ' + q[q.index(resume):] if resume else '')
    if choices:
        q += ' [' + choices.group(1) + ']'
    return q


def tex(s):
    """Escape for LaTeX, keeping the paper's typography for quotes and dashes."""
    s = re.sub(r'"([^"]*)"', r"``\1''", s)
    s = s.replace('i.e. ', r'i.e.\ ').replace(' -- ', ' -- ')
    return s.replace('_', r'\_').replace('%', r'\%').replace('&', r'\&')

## Load every question, with the figure types it is asked of

In [3]:
records = {}                                  # vqa_id -> record
asked_of = collections.defaultdict(set)       # question key -> figure types
text_of = collections.defaultdict(set)        # question key -> distinct texts

for fp in sorted(glob.glob(os.path.join(ex(qa_jsons_dir), '*_qa.json'))):
    rec = json.loads(json.load(open(fp)))
    p = rec['plot0']
    cat = ('contour' if p['type'] == 'contour'
           else 'sky-gmm' if p.get('distribution') == 'gmm' else 'sky-real')
    records[rec['vqa_id']] = rec
    for name, e in entries(rec):
        asked_of[name].add(cat)
        text_of[name].add(e['question'].strip())

published = {}
for fp in sorted(glob.glob(os.path.join(ex(realsky_dir), 'qa_jsons', '*_qa.json'))):
    rec = json.loads(json.load(open(fp)))
    published[rec['vqa_id']] = records[rec['vqa_id']] = rec
    for name, e in entries(rec):
        asked_of[name].add('published')
        text_of[name].add(e['question'].strip())

print('%d of our figures, %d published' % (len(records) - len(published), len(published)))
for name in sorted(asked_of):
    flag = '' if len(text_of[name]) == 1 else '   <-- %d wordings' % len(text_of[name])
    print('  %-40s %s%s' % (name, sorted(asked_of[name]), flag))

2001 of our figures, 104 published
  aspect ratio                             ['contour', 'sky-gmm', 'sky-real']
  center declination                       ['published']
  center right ascension                   ['published']
  colormap                                 ['contour', 'sky-gmm', 'sky-real']
  distribution-color + list)               ['contour']
  distribution-image + list)               ['sky-gmm', 'sky-real']
  distribution-x/y + list)                 ['contour']
  epoch                                    ['sky-gmm', 'sky-real']
  field height                             ['published', 'sky-gmm', 'sky-real']
  field width                              ['published', 'sky-gmm', 'sky-real']
  finest unit declination                  ['sky-gmm', 'sky-real']
  finest unit right ascension              ['sky-gmm', 'sky-real']
  image or lines (list)                    ['contour']
  maximum color                            ['contour', 'sky-gmm', 'sky-real']
  maximum declination ax

## Table 1 -- example questions

"Applies to" is computed from the data: `all plots` when a question is asked
of every one of our figure types, otherwise the types it is asked of (plus
`published` when the published set asks the identically worded question).

In [4]:
def applies_to(name):
    cats = asked_of[name]
    ours = cats - {'published'}
    if ours == {'contour', 'sky-gmm', 'sky-real'}:
        label = ['all plots']
    else:
        label = [CATEGORY_NAMES[c] for c in ('contour', 'sky-gmm', 'sky-real') if c in ours]
    if 'published' in cats:
        label.append('published')
    return ', '.join(label)


rows, prev_level = [], None
for level, name, vid, cut, fmt in table1_rows:
    e = dict(entries(records[vid]))[name]
    if prev_level is not None and level != prev_level:
        rows.append(r'\midrule')
    prev_level = level
    rows.append('%d & %s & %s & %s \\\\' % (level, tex(question_text(e, cut)), applies_to(name),
                                            tex(fmt(answer(e, name)))))

examples = sorted({vid for _, _, vid, _, _ in table1_rows})
table1 = r"""\begin{table*}[t]
\centering\small\color{red}
\begin{tabular}{@{}c p{0.56\textwidth} p{0.16\textwidth} p{0.17\textwidth}@{}}
\toprule
Lvl & Question [choices offered] & Applies to & Example Answer \\
\midrule
%s
\bottomrule
\end{tabular}
\caption{Example questions for each level, the plot types they are asked of, and an example ground-truth answer. Choices offered are given in brackets; long prompts are abridged ([...]). Plot types: contour, GMM-sky and real-sky are our figures; published are the human-verified Astronomy Image Explorer figures. Example answers come from single figures of each type (\textsf{vqa\_000001}, GMM-sky; \textsf{vqa\_000002} and \textsf{vqa\_000004}, real-sky; \textsf{vqa\_000007}, contour; \textsf{rsky\_000001}, published); numeric answers are in the units the question asks for.}
\label{tab:questions}
\end{table*}""" % '\n'.join(rows)

open(os.path.join(ex(tab_out_dir), 'table_example_questions.tex'), 'w').write(table1 + '\n')
print(table1)

\begin{table*}[t]
\centering\small\color{red}
\begin{tabular}{@{}c p{0.56\textwidth} p{0.16\textwidth} p{0.17\textwidth}@{}}
\toprule
Lvl & Question [choices offered] & Applies to & Example Answer \\
\midrule
1 & What is the plot type of the plot in this figure? [contour, image of the sky] & all plots & image of the sky \\
1 & What is the colormap that was used in this figure? & all plots & rainbow\_r \\
1 & What coordinate epoch is given on the axis labels of this figure? Answer ``none'' if no epoch is stated. & GMM-sky, real-sky & J2000 \\
1 & What is the smallest unit of angle shown on the right ascension tick labels of this figure? [hours, minutes, seconds] & GMM-sky, real-sky & minutes \\
1 & What is the minimum right ascension value covered by the right ascension axis of this figure -- that is, the lower limit of the axis range, not of the data? Give the value in hours, minutes and seconds [...] & GMM-sky, real-sky & 18h46m50.98s \\
1 & What is the style of the contour plot? [ima

## Table 2 -- questions asked of the published figures

Every question Gemini was asked about the published figures, read from its run
pickles, joined to the released ground truth.  `pixel scale` was asked but is
not released or scored: the published astrometry counts pixels of the page
scan, ours counts data pixels (see the dataset README).

In [5]:
# what Gemini was actually asked, by question text
asked = collections.Counter()
for fp in glob.glob(os.path.join(ex(published_run), '*_qa.pickle')):
    p = pickle.load(open(fp, 'rb'))
    for e in (p[0] if isinstance(p, (list, tuple)) else p):
        asked[(e.get('question') or '').strip()] += 1

# the released questions, their level and ground-truth range
released = collections.OrderedDict()
for vid, rec in published.items():
    for lvl, kinds in rec['VQA'].items():
        for items in kinds.values():
            for name, e in items.items():
                d = released.setdefault(name, {'level': int(lvl.split()[-1]), 'entry': e, 'values': []})
                d['values'].append(float(answer(e, name)))

GROUND_TRUTH = {
    'field width':  'mean length of the plot box\'s top and bottom edges on the sky',
    'field height': 'mean length of the plot box\'s left and right edges on the sky',
    'center right ascension': 'sky position of the plot-box centre',
    'center declination':     'sky position of the plot-box centre',
}
UNITS = {'field width': "$'$", 'field height': "$'$",              # arcmin
         'center right ascension': r'$^\circ$', 'center declination': r'$^\circ$'}
CUT = {'center right ascension': ('J2000 (FK5) frame?', 'Give the value'),
       'center declination':     ('J2000 (FK5) frame?', 'Give the value')}

def num(x):
    """Range endpoint: whole numbers from 10 up, two decimals below; TeX minus."""
    t = ('%.0f' % x) if abs(x) >= 10 else ('%.2f' % x)
    return t.replace('-', '$-$')


rows2 = []
for name, d in released.items():
    e, v = d['entry'], np.array(d['values'])
    shared = 'yes' if len(text_of[name]) == 1 and asked_of[name] - {'published'} else 'no'
    rng = '%s--%s%s' % (num(v.min()), num(v.max()), UNITS[name])
    n_asked = asked.get(e['question'].strip(), 0)
    rows2.append('%d & %s & %s & %s & %s \\\\' % (d['level'], tex(question_text(e, CUT.get(name))),
                                                  GROUND_TRUTH[name], rng, shared))
    print('%-24s asked of %d figures; wording shared with our figures: %s' % (name, n_asked, shared))

# pixel scale: asked, then dropped
pix = [q for q in asked if q.startswith('Approximately what angular size')]
assert len(pix) == 1
rows2.append(r'\midrule')
rows2.append('3 & %s & not released: scan pixels, not data pixels & -- & %s \\\\'
             % (tex(pix[0]), 'yes' if pix[0] in text_of['pixel scale'] else 'no'))

n_pub = len(published)
table2 = r"""\begin{table*}[t]
\centering\small\color{red}
\begin{tabular}{@{}c p{0.42\textwidth} p{0.24\textwidth} p{0.13\textwidth} c@{}}
\toprule
Lvl & Question & Ground truth & Range & Shared \\
\midrule
%s
\bottomrule
\end{tabular}
\caption{Questions asked of the %d human-verified published figures (\textsf{Gemini} only). Ground truth is measured through each figure's AVM astrometry over its human-verified plot box (\autoref{app:angular}); the range is over all %d figures, in arcminutes ($'$) or degrees. ``Shared'' marks questions worded identically to those asked of our figures, so their scores are directly comparable. Questions about other figure elements (colormap, colorbar statistics, provenance) have no ground truth in the metadata and are not asked; pixel scale was asked but is not scored.}
\label{tab:published_questions}
\end{table*}""" % ('\n'.join(rows2), n_pub, n_pub)

open(os.path.join(ex(tab_out_dir), 'table_published_questions.tex'), 'w').write(table2 + '\n')
print()
print(table2)

field width              asked of 104 figures; wording shared with our figures: yes
field height             asked of 104 figures; wording shared with our figures: yes
center right ascension   asked of 104 figures; wording shared with our figures: no
center declination       asked of 104 figures; wording shared with our figures: no

\begin{table*}[t]
\centering\small\color{red}
\begin{tabular}{@{}c p{0.42\textwidth} p{0.24\textwidth} p{0.13\textwidth} c@{}}
\toprule
Lvl & Question & Ground truth & Range & Shared \\
\midrule
2 & What is the angular width of the sky region shown in this figure, measured along the right ascension axis, as a true angle on the sky (i.e.\ including the cos(declination) factor)? Give the value in arcminutes. & mean length of the plot box's top and bottom edges on the sky & 0.33--957$'$ & yes \\
2 & What is the angular height of the sky region shown in this figure, measured along the declination axis? Give the value in arcminutes. & mean length of the plot box

## Appendix C numbers -- do the two angular-width definitions agree?

Our figures: width = (RA range of the displayed region) × cos(mid-declination)
(`sky_plot_qa_utils.field_extent`).  Published figures: width = mean on-sky
length of the plot box's top and bottom edges, through a TAN projection built
from the AVM metadata (`real_data/scripts/apply_plot_boxes.py`).

Both are recomputed here for every published figure, from its stored plot-box
corners and AVM parameters.  The first check confirms the released ground
truth is reproduced; the second measures how far the two definitions differ.

In [6]:
from astropy.wcs import WCS
from astropy.coordinates import SkyCoord


def avm_wcs(wwt):
    """TAN WCS from the AVM/WWT parameters, as in apply_plot_boxes.py."""
    scale = float(wwt['scale']) / 3600.0
    pa = np.deg2rad(float(wwt.get('rotation', 180.0)) - 180.0)
    w = WCS(naxis=2)
    w.wcs.ctype = ['RA---TAN', 'DEC--TAN']
    w.wcs.crval = [float(wwt['ra']), float(wwt['dec'])]
    w.wcs.crpix = [float(wwt['x']) + 0.5, float(wwt['y']) + 0.5]
    w.wcs.cd = [[-scale * np.cos(pa), scale * np.sin(pa)],
                [scale * np.sin(pa), scale * np.cos(pa)]]
    return w


rows3 = []
for vid, rec in published.items():
    H = rec['image']['height']
    w = avm_wcs(rec['source']['wwt'])
    tl, tr, br, bl = (np.array(p, float) for p in rec['source']['plot_box']['corners_xy_topleft'])
    to_sky = lambda pts: w.all_pix2world(np.c_[pts[:, 0] + 0.5, H - pts[:, 1] + 0.5], 1)

    # (a) the published definition: great-circle edge lengths
    c = SkyCoord(*to_sky(np.array([tl, tr, br, bl])).T, unit='deg')
    edges = 0.5 * (c[0].separation(c[1]) + c[3].separation(c[2])).arcmin

    # (b) our definition: sample the box, RA range x cos(mid-dec)
    s = np.linspace(0, 1, 64)
    top = tl + np.outer(s, tr - tl)
    bot = bl + np.outer(s, br - bl)
    grid = np.concatenate([top + np.outer(t, np.ones(2)) * (bot - top) for t in s])
    ra, dec = to_sky(grid).T
    ra = np.degrees(np.unwrap(np.radians(ra)))
    cosdec = (ra.max() - ra.min()) * np.cos(np.radians(0.5 * (dec.min() + dec.max()))) * 60

    rows3.append({'vqa_id': vid, 'released': float(answer(dict(entries(rec))['field width'], 'field width')),
                  'edges': edges, 'cosdec': cosdec, 'dec': 0.5 * (dec.min() + dec.max())})

import pandas as pd
wd = pd.DataFrame(rows3)
wd['rel_diff_%'] = 100 * (wd.cosdec / wd.edges - 1)
print('released ground truth reproduced by (a): max |difference| = %.3f%%'
      % (100 * (wd.edges / wd.released - 1).abs().max()))
d = wd['rel_diff_%'].abs()
print('(b) vs (a): median %.1f%%, 90th percentile %.1f%%, within 2%%: %d of %d, max %.0f%%'
      % (d.median(), d.quantile(0.9), (d < 2).sum(), len(d), d.max()))
print('\nlargest differences:')
print(wd.reindex(d.sort_values(ascending=False).index).head(5)
        .assign(width_deg=lambda x: x.edges / 60).round(2).to_string(index=False))

released ground truth reproduced by (a): max |difference| = 0.013%
(b) vs (a): median 0.4%, 90th percentile 1.9%, within 2%: 94 of 104, max 23%

largest differences:
     vqa_id  released  edges  cosdec    dec  rel_diff_%  width_deg
rsky_000132    956.81 956.81 1177.51  57.79       23.07      15.95
rsky_000064    448.07 448.06  495.13  20.56       10.50       7.47
rsky_000033    193.11 193.11  203.91 -67.27        5.60       3.22
rsky_000032    194.58 194.57  205.37 -67.27        5.55       3.24
rsky_000055    428.37 428.37  447.27  36.20        4.41       7.14
